# Kidney_UAE Data Exploration

## Load Requisite Libraries

In [ ]:
import sys
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


from core.functions import *  # import custom functions
from core.constants import var_index

## Read File From Path and Explore Basic Structure

In [ ]:
# Change directory to where functions.py is located if it's not in '/content'
data_path = "../data/processed/"

In [ ]:
# read in the data from an excel file
df = pd.read_parquet(os.path.join(data_path, "df_sans_zero.parquet")).set_index(var_index)

In [ ]:
print(f"There are {df.shape[0]} rows and {df.shape[1]} columns in this dataset.")

In [ ]:
df.head()  # print first 5 rows of dataframe

## Create EDA Dataset

In [ ]:
df_eda = df.copy(deep=True) # create new dataframe specifically for EDA
df_eda["time_years"] = round(df_eda["time_months"] / 12, 1)

In [ ]:
sex_map = {0: "Female", 1: "Male"}
outcome_map = {0: "No Death", 1: "Death"}
df_eda["outcome"] = df_eda["outcome"].map(outcome_map)
df_eda["sex"] = df_eda["sex"].map(sex_map)

In [ ]:
sns.heatmap(pd.crosstab(df_eda["sex"], df_eda["outcome"]), annot=True, fmt="d", cmap="YlGnBu")
plt.xlabel("Outcome")
plt.ylabel("Sex")
plt.title("Crosstab of Sex and Outcome")
plt.show()

In [ ]:
# Define bins so that there's a clear bin for > 10 up to max
# (and potentially slightly beyond)
# Note: The last bin captures all values from 10.0 up to and including max and
# slightly beyond, if necessary
year_bins = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, float("inf")]
year_labels = [
    "0-1_years",
    "1-2_years",
    "2-3_years",
    "3-4_years",
    "4-5_years",
    "5-6_years",
    "6-7_years",
    "7-8_years",
    "8-9_years",
    "9-10_years",
    "10_years_plus",
]

# Apply the binning
df_eda["year_bins"] = pd.cut(
    df_eda["time_years"],
    bins=year_bins,
    labels=year_labels,
    include_lowest=True,
    right=True,
)

In [ ]:
# create separate dataframe for expanded modeling with one-hot-encoded year bins
df_years = (
    df_eda.copy(deep=True)
    .assign(**pd.get_dummies(df_eda["year_bins"]))
    .drop(columns=["time_months", "time_years", "year_bins"])
)

In [ ]:
df_years

## References

Al-Shamsi, S., Govender, R. D., & King, J. (2021). Predictive value of creatinine-based equations of kidney function in the long-term prognosis of United Arab Emirates patients with vascular risk. *Oman medical journal, 36*(1), e217. https://doi.org/10.5001/omj.2021.07


Al-Shamsi, S., Govender, R. D., & King, J. (2019). Predictive value of creatinine-based equations of kidney function in the long-term prognosis of United Arab Emirates patients with vascular risk [Dataset]. Mendeley Data, V1. https://data.mendeley.com/datasets/ppfwfpprbc/1



